# Reshape H1 2024 into a monthly table

*Data Analysis in Python · Topic 4.5*

The finance team wants January to June 2024 as a spreadsheet-style table: months down the side, channels across the top, with sessions and revenue. The analytics team wants the same numbers long, to chart them.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell, the cell that builds the `mini` table (four rows typed in), the cell that builds the `sessions` table (lesson 3.'s steps) and the cell that adds its `channel_group` column (topic 4.2.'s steps).
2. In the first cell marked ✏️, create two tables:
   - `mini_wide`: the `mini` table with months as rows and channel groups as columns, filled with sessions
   - `mini_long`: the `mini_wide` table back as a long table, with the columns `month`, `channel_group` and `sessions`
3. In the second cell marked ✏️, add a `month` column to the `sessions` table, such as `"2024-03"`, and create the `monthly` table: one row per month and channel group, with the columns `month`, `channel_group`, `sessions`, `transactions` and `revenue`.
4. In the third cell marked ✏️, create two tables:
   - `wide_sessions`: months as rows, channel groups as columns, filled with each cell's total sessions, made from the `sessions` table with `pivot_table()`
   - `long_sessions`: the `wide_sessions` table as a long table, with the columns `month`, `channel_group` and `sessions`
5. In the last cell marked ✏️, create the `report` table: months as rows, and a column for each channel group's sessions and each one's revenue, named like `"Email sessions"` and `"Email revenue"`.
6. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: The `mini_wide` table has two rows and two columns, and the `mini_long` table four rows. The `monthly` table has 54 rows. The `wide_sessions` table has six months and nine channel groups, with 21,375 Organic Search sessions in March 2024, and adds up to 464,344. The `long_sessions` table has 54 rows. The `report` table has 18 columns, from `Direct sessions` to `Referral revenue`, and Email's March revenue is $35,860.42.

## Parameters

In [ ]:
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = ["2024-01", "2024-02", "2024-03", "2024-04", "2024-05", "2024-06"]
START, END = "2024-01-01", "2024-06-30"

In [ ]:
mini = pd.DataFrame({
    "month": ["2024-01", "2024-01", "2024-02", "2024-02"],
    "channel_group": ["Email", "Paid Search", "Email", "Paid Search"],
    "sessions": [500, 800, 450, 900],
})
mini

In [ ]:
# The clean sessions table, with lesson 3's steps. Run it, and leave it as it is.
DAY_FIRST_MONTHS = ["2025-03"]
SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}


def read_export(month):
    """Read one month's export with dates as dates, revenue as numbers and lost campaigns as missing."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    return export


raw = pd.concat([read_export(month) for month in MONTHS], ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["source"] = sessions["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
sessions.shape

In [ ]:
# Channel groups, with topic 4.2's steps and Foundations topic 12.3's function. Run it, and leave it as it is.
SOCIAL_SITES = {"facebook.com", "m.facebook.com", "l.facebook.com", "lm.facebook.com", "instagram.com", "l.instagram.com"}


def clean(value):
    """Return the value lowercased and stripped, or "" for None."""
    if value is None:
        return ""
    return value.strip().lower()


def channel_group(source, medium):
    """Return the channel group of a session's source and medium, with Foundations topic 12.3's rules."""
    source = clean(source)
    medium = clean(medium)
    if source == "(direct)" and medium == "(none)":
        return "Direct"
    if medium == "email":
        return "Email"
    if medium == "cpc":
        return "Paid Search"
    if medium == "display":
        return "Display"
    if medium == "paid_social":
        return "Paid Social"
    if medium == "video":
        return "Paid Video"
    if medium == "organic":
        return "Organic Search"
    if medium == "referral" and source in SOCIAL_SITES:
        return "Organic Social"
    if medium == "referral":
        return "Referral"
    return "Unassigned"


def row_channel_group(row):
    """Return the channel group of one row of the sessions table."""
    return channel_group(row["session_source"], row["session_medium"])


sessions["channel_group"] = sessions.apply(row_channel_group, axis=1)
sessions["channel_group"].value_counts()

## Four rows

In [ ]:
mini_wide = mini.pivot(index="month", columns="channel_group", values="sessions")
mini_long = mini_wide.reset_index().melt(id_vars="month", var_name="channel_group", value_name="sessions")
mini_wide, mini_long

## Six months

In [ ]:
sessions["month"] = sessions["date"].dt.strftime("%Y-%m")
monthly = sessions.groupby(["month", "channel_group"], as_index=False)[["sessions", "transactions", "revenue"]].sum()
monthly.head()

In [ ]:
wide_sessions = sessions.pivot_table(index="month", columns="channel_group", values="sessions", aggfunc="sum")
long_sessions = wide_sessions.reset_index().melt(id_vars="month", var_name="channel_group", value_name="sessions")
wide_sessions.sum().sum() == sessions["sessions"].sum(), long_sessions["sessions"].sum() == sessions["sessions"].sum()

In [ ]:
report = monthly.pivot(index="month", columns="channel_group", values=["sessions", "revenue"])
report.columns = [f"{group} {metric}" for metric, group in report.columns]
report

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    mini_wide = found.get("mini_wide")
    mini_long = found.get("mini_long")
    if not isinstance(mini_wide, pd.DataFrame):
        results.append((False, "mini_wide isn't a DataFrame yet: mini.pivot(index=\"month\", columns=\"channel_group\", values=\"sessions\")."))
    elif mini_wide.shape != (2, 2) or list(mini_wide.index) != ["2024-01", "2024-02"] or mini_wide.loc["2024-02", "Paid Search"] != 900:
        results.append((False, "mini_wide should have the two months as rows, Email and Paid Search as columns, and 900 Paid Search sessions in February."))
    elif not isinstance(mini_long, pd.DataFrame) or len(mini_long) != 4:
        results.append((False, "mini_long should be mini_wide as four rows again: mini_wide.reset_index().melt(id_vars=\"month\", var_name=\"channel_group\", value_name=\"sessions\")."))
    elif sorted(mini_long.columns) != ["channel_group", "month", "sessions"]:
        results.append((False, f"mini_long's columns should be month, channel_group and sessions. They are {list(mini_long.columns)}: name them with var_name and value_name."))
    else:
        results.append((True, "mini went wide and came back long, with the same four numbers."))

    sessions = found.get("sessions")
    monthly = found.get("monthly")
    if not isinstance(sessions, pd.DataFrame) or len(sessions) != 10181 or sessions["sessions"].sum() != 464344:
        results.append((False, "sessions should be the clean table for January to June 2024, 10,181 rows. Run the cells that build it, and don't filter it."))
    elif "month" not in sessions.columns or sessions["month"].nunique() != 6:
        results.append((False, "sessions has no month column with the six months yet: sessions[\"month\"] = sessions[\"date\"].dt.strftime(\"%Y-%m\")."))
    elif not isinstance(monthly, pd.DataFrame) or len(monthly) != 54:
        results.append((False, "monthly should have 54 rows, one per month and channel group: sessions.groupby([\"month\", \"channel_group\"], as_index=False)[[\"sessions\", \"transactions\", \"revenue\"]].sum()."))
    elif not {"month", "channel_group", "sessions", "transactions", "revenue"} <= set(monthly.columns):
        results.append((False, f"monthly should have month and channel_group as columns, with sessions, transactions and revenue. Its columns are {list(monthly.columns)}: group with as_index=False."))
    elif monthly["sessions"].sum() != 464344:
        results.append((False, f"monthly's sessions add up to {monthly['sessions'].sum():,}, but sessions has 464,344. Total each group with sum()."))
    else:
        results.append((True, "monthly has the 54 month and channel rows, and its sessions add up to 464,344."))

    wide = found.get("wide_sessions")
    long = found.get("long_sessions")
    if not isinstance(wide, pd.DataFrame):
        results.append((False, "wide_sessions isn't a DataFrame yet: sessions.pivot_table(index=\"month\", columns=\"channel_group\", values=\"sessions\", aggfunc=\"sum\")."))
    elif wide.shape != (6, 9):
        results.append((False, f"wide_sessions should have six months as rows and nine channel groups as columns. Its shape is {wide.shape}."))
    elif abs(wide.loc["2024-01", "Direct"] - 307.9) < 0.01:
        results.append((False, "wide_sessions' January Direct is 307.9: pivot_table() averaged the rows. Pass aggfunc=\"sum\"."))
    elif wide.loc["2024-03", "Organic Search"] != 21375 or wide.sum().sum() != 464344:
        results.append((False, f"wide_sessions should total each month and channel's sessions: 21,375 for Organic Search in March, and 464,344 in all. It has {wide.loc['2024-03', 'Organic Search']:,} and {wide.sum().sum():,}."))
    elif not isinstance(long, pd.DataFrame) or len(long) != 54:
        results.append((False, "long_sessions should be wide_sessions as 54 rows: wide_sessions.reset_index().melt(id_vars=\"month\", var_name=\"channel_group\", value_name=\"sessions\")."))
    elif sorted(long.columns) != ["channel_group", "month", "sessions"] or long["sessions"].sum() != 464344:
        results.append((False, f"long_sessions should have the columns month, channel_group and sessions, adding up to 464,344. Its columns are {list(long.columns)}."))
    else:
        results.append((True, "wide_sessions has six months and nine channels, 21,375 Organic Search sessions in March, and long_sessions has the same 464,344 in 54 rows."))

    report = found.get("report")
    groups = ["Direct", "Display", "Email", "Organic Search", "Organic Social", "Paid Search", "Paid Social", "Paid Video", "Referral"]
    expected = [f"{group} {metric}" for metric in ["sessions", "revenue"] for group in groups]
    if not isinstance(report, pd.DataFrame):
        results.append((False, "report isn't a DataFrame yet. Pivot monthly with values=[\"sessions\", \"revenue\"], then flatten the column names."))
    elif isinstance(report.columns, pd.MultiIndex):
        results.append((False, "report's column names still have two levels, such as ('sessions', 'Email'). Flatten them: report.columns = [f\"{group} {metric}\" for metric, group in report.columns]."))
    elif "sessions Email" in report.columns:
        results.append((False, "report's columns are named like \"sessions Email\". Put the channel first: f\"{group} {metric}\" for metric, group in report.columns."))
    elif sorted(report.columns) != sorted(expected) or len(report) != 6:
        results.append((False, f"report should have six months and 18 columns, from \"Direct sessions\" to \"Referral revenue\". It has {len(report)} rows and the columns {list(report.columns)[:4]}..."))
    elif abs(report.loc["2024-03", "Email revenue"] - 35860.42) > 0.005 or report.loc["2024-03", "Email sessions"] != 7540:
        results.append((False, f"report's March Email should be 7,540 sessions and $35,860.42 in revenue. It has {report.loc['2024-03', 'Email sessions']:,} and {report.loc['2024-03', 'Email revenue']:,.2f}: pivot monthly, which has one row per cell."))
    else:
        results.append((True, "report has one row of column names, from Direct sessions to Referral revenue: Email's March is 7,540 sessions and $35,860.42."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
mini_wide = mini.pivot(index="month", columns="channel_group", values="sessions")
mini_long = mini_wide.reset_index().melt(id_vars="month", var_name="channel_group", value_name="sessions")
mini_wide, mini_long
```

```python
sessions["month"] = sessions["date"].dt.strftime("%Y-%m")
monthly = sessions.groupby(["month", "channel_group"], as_index=False)[["sessions", "transactions", "revenue"]].sum()
monthly.head()
```

```python
wide_sessions = sessions.pivot_table(index="month", columns="channel_group", values="sessions", aggfunc="sum")
long_sessions = wide_sessions.reset_index().melt(id_vars="month", var_name="channel_group", value_name="sessions")
wide_sessions.sum().sum() == sessions["sessions"].sum(), long_sessions["sessions"].sum() == sessions["sessions"].sum()
```

```python
report = monthly.pivot(index="month", columns="channel_group", values=["sessions", "revenue"])
report.columns = [f"{group} {metric}" for metric, group in report.columns]
report
```

</details>